# Import Libraries and Set Seeds

Import necessary libraries including torch, random, numpy, pathlib, typing, os, wandb, logging, datetime, and csv. Set random seeds for reproducibility.

In [ ]:
import torch, random, numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Any
import os 
import wandb
import logging
import datetime
import csv
import sys
sys.path.append("..")



from manager import AnomalyDetectionManager as ADM

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

# Define Helper Functions

Define the getResults function to read and parse metric results from CSV files based on dataset, model, and tiling options.

In [4]:
def getResults(resultsFolder:Path, datasetName:str, modelName:str, tiled:bool) -> Dict[str,float]:
    resultsPath: Path = resultsFolder / datasetName / modelName 
    if tiled:
        resultsPath  = resultsPath / "tiled"
    resultsPath = resultsPath / "metricResults.csv"
    results:Dict[str,float] = {}
    with open(resultsPath, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            auroc = float(row["image_AUROC"])
            f1   = float(row["image_F1Score"])
            aupr = float(row["image_AUPR"])
            results["image_AUROC"] = auroc
            results["image_F1Score"] = f1
            results["image_AUPR"] = aupr
    return results

# Setup Environment Variables and Paths

Set environment variables for FiftyOne MongoDB URI and Wandb API key. Define paths for configs, tiling, models, datasets, and results folders.

In [5]:
# Point FiftyOne at your MongoDB instance
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost"
os.environ["WANDB_API_KEY"] = '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'

logger = logging.getLogger("logger")
configDir:Path = Path("../configs")

tilingDir:Path = Path("../configs/Tiling")
tilingConfigPath = tilingDir / "TiledEnsemble.yaml"

modelDir:Path = Path("../configs/Models")
modelList: List[str] = ["padim", "patchcore", "fastflow", "reverse_distillation", "efficientad-m", "efficientad-s"]
modelPath:Path = modelDir / Path(modelList[0] + ".yaml")

datasetDir:Path = Path("../datasets")
datasetName:str = "MVTecAD"
datasetPath:Path = datasetDir / datasetName

category = "all"

resultsFolder:Path = Path("resultsComparison")

tiling:bool = True
train:bool = False
evaluate:bool = True

# Initialize Logging and Wandb

Initialize the logger and attempt to log in to Wandb.

In [6]:
# Try to setup wandb
wandb.login()

wandb: Currently logged in as: daniel-pommer (daniel-pommer-technische-hochschule-n-rnberg-georg-simon-ohm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Initialize IAD and Load Dataset

Create an IAD instance, load the dataset from disk, select the category, and adjust the output path.

In [ ]:
manager = ADM("comparsionPipeline.log", resultsFolder, configDir=configDir, datasetDir=datasetDir)

manager.loadDatasetFromDisk(datasetPath, datasetName, overwrite=True, merge=False)
manager.selectCategory(category)
manager.adjustOutputPath()

Subprocess ['/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/fiftyone/db/bin/mongod', '--dbpath', '/Users/dapo/.fiftyone/var/lib/mongo', '--logpath', '/Users/dapo/.fiftyone/var/lib/mongo/log/mongo.log', '--port', '0', '--nounixsocket'] exited with error 100:


{"t":{"$date":"2026-03-16T12:26:22.095Z"},"s":"I",  "c":"CONTROL",  "id":20697,   "ctx":"-","msg":"Renamed existing log file","attr":{"oldLogPath":"/Users/dapo/.fiftyone/var/lib/mongo/log/mongo.log","newLogPath":"/Users/dapo/.fiftyone/var/lib/mongo/log/mongo.log.2026-03-16T12-26-22"}}


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 iad = IAD("comparsionPipeline.log", resultsFolder, configDir=configDir, datasetDir=datas     │
│   2                                                                                              │
│ ❱ 3 iad.loadDatasetFromDisk(datasetPath, datasetName, overwrite=True, merge=False)               │
│   4 iad.selectCategory(category)                                                                 │
│   5 iad.adjustOutputPath()                                                                       │
│   6                                                                                              │
│                                                                                                  │
│ /Users/dapo/Documents/Code/IAD/src/IAD.py:320 in loadDatasetFromDisk                             │
│                                                                                                  │
│   317 │   │   │   │   overwrite = False                                                          │
│   318 │   │   │   elif not datasetPath.exists():                                                 │
│   319 │   │   │   │   FileNotFoundError(f"Dataset {datasetPath} does not exit")                  │
│ ❱ 320 │   │   │   elif fo.dataset_exists(datasetName) and not overwrite and not merge:           │
│   321 │   │   │   │   logger.info(f"Dataset '{datasetName}' already exists in database")         │
│   322 │   │   │   │   logger.info("Loading from database")                                       │
│   323 │   │   │   │   self.loadDatasetFromDatabase(datasetName)                                  │
│                                                                                                  │
│ /Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/fiftyone/core/dataset.py:104   │
│ in dataset_exists                                                                                │
│                                                                                                  │
│     101 │   Returns:                                                                             │
│     102 │   │   True/False                                                                       │
│     103 │   """                                                                                  │
│ ❱   104 │   conn = foo.get_db_conn()                                                             │
│     105 │   return bool(list(conn.datasets.find({"name": name}, {"_id": 1}).limit(1)))           │
│     106                                                                                          │
│     107                                                                                          │
│                                                                                                  │
│ /Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/fiftyone/core/odm/database.py: │
│ 470 in get_db_conn                                                                               │
│                                                                                                  │
│    467 │   Returns:                                                                              │
│    468 │   │   a ``pymongo.database.Database``                                                   │
│    469 │   """                                                                                   │
│ ❱  470 │   _connect()                                                                            │
│    471 │   db = _client[fo.config.database_name]                                                 │
│    472 │   return _apply_options(db)                       

# Setup Tiling Configuration

If tiling is enabled, set up the tiling configuration using the specified YAML file.

In [ ]:
if tiling:
    manager.setupTiling(configDir / Path("Tiling/TiledEnsemble.yaml"))

# Initialize Results CSV File

Create and initialize the accumulated results CSV file with headers including Date, Dataset, Model, and metrics.

In [ ]:
compResultsFile:Path = resultsFolder / "accumulatedResults.csv"
columns = ["Date", "Dataset", "Model", "image_AUROC", "image_F1Score", "image_AUPR"]
date = datetime.date.today()
with open(compResultsFile, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(columns)

# Loop Over Models for Training and Evaluation

Iterate over the list of models: generate the model, train it with the appropriate config, load checkpoint if available, evaluate, retrieve results, and append to the CSV file.

In [ ]:
for modelName in modelList:
    manager.generateModel(f"{modelName}.yaml")
    if train:
        manager.train(configDir / Path(f"Trainer/Training_{modelName}.yaml"), tiling=tiling)
    if evaluate:
        if manager.ckptPath is not None:
            if not tiling:
                manager.loadCheckpoint(manager.ckptPath, f"{modelName}")
            manager.eval(configDir / "Trainer" / "Evaluation.yaml", tiling=tiling)
    
    try:
        results = getResults(resultsFolder=resultsFolder, datasetName=datasetName, modelName=modelName, tiled=tiling)

        with open(compResultsFile, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                str(date),
                str(datasetName),
                str(modelName),
                str(results["image_AUROC"]),
                str(results["image_F1Score"]),
                str(results["image_AUPR"])])
    except:
        logger.error(f"Could not write to results file for model {modelName} and dataset {datasetName}")

INFO: Model Padim loaded: {'model': {'class_path': 'Padim', 'init_args': {'backbone': 'resnet18', 'layers': ['layer1', 'layer2', 'layer3'], 'pre_trained': True, 'n_features': 100, 'pre_processor': PreProcessor(), 'post_processor': AOIPostProcessor(
  (_image_threshold_metric): F1AdaptiveThreshold()
  (_pixel_threshold_metric): F1AdaptiveThreshold()
  (_image_min_max_metric): MinMax()
  (_pixel_min_max_metric): MinMax()
), 'visualizer': False, 'evaluator': Evaluator(
  (val_metrics): ModuleList(
    (0): AUROC()
  )
  (test_metrics): ModuleList(
    (0): AUROC()
    (1): F1Score()
    (2): AUPR()
  )
)}}}
INFO: Initializing Padim model.
INFO: Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k)


/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'post_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['post_processor'])`.
/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'evaluator' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['evaluator'])`.


INFO: [timm/resnet18.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO: Missing keys (fc.weight, fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
INFO: Model Padim(
  (pre_processor): PreProcessor()
  (post_processor): AOIPostProcessor(
    (_image_threshold_metric): F1AdaptiveThreshold()
    (_pixel_threshold_metric): F1AdaptiveThreshold()
    (_image_min_max_metric): MinMax()
    (_pixel_min_max_metric): MinMax()
  )
  (evaluator): Evaluator(
    (val_metrics): ModuleList(
      (0): AUROC()
    )
    (test_metrics): ModuleList(
      (0): AUROC()
      (1): F1Score()
      (2): AUPR()
    )
  )
  (model): PadimModel(
    (feature_extractor): TimmFeatureExtractor(
      (feature_extractor): FeatureListNet(
        (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:9                                                                                    │
│                                                                                                  │
│    6 │   │   if iad.ckptPath is not None:                                                        │
│    7 │   │   │   if not tiling:                                                                  │
│    8 │   │   │   │   iad.loadCheckpoint(iad.ckptPath, f"{modelName}")                            │
│ ❱  9 │   │   │   iad.eval(configDir / "Trainer" / "Evaluation.yaml", tiling=tiling)              │
│   10 │                                                                                           │
│   11 │   try:                                                                                    │
│   12 │   │   results = getResults(resultsFolder=resultsFolder, datasetName=datasetName, model    │
│                                                                                                  │
│ /Users/dapo/Documents/Code/IAD/src/IAD.py:771 in eval                                            │
│                                                                                                  │
│   768 │   │   else:                                                                              │
│   769 │   │   │   self.runName =f"{self.modelName}-{self.datasetName}"                           │
│   770 │   │                                                                                      │
│ ❱ 771 │   │   self.adjustOutputPath()                                                            │
│   772 │   │   self.setupLogging()                                                                │
│   773 │   │   self.setupTrainingCallbacks()                                                      │
│   774 │   │   self.setupWandBLogger(self.runName, self.outputPath, self.version)                 │
│                                                                                                  │
│ /Users/dapo/Documents/Code/IAD/src/IAD.py:794 in _evalTiledModel                                 │
│                                                                                                  │
│   791 │   │   self._checkBeforeTraining()                                                        │
│   792 │   │   # Setup datamodule for the tiling                                                  │
│   793 │   │   self.datamoduleParams = {key: evalConfig[key] for key in DATAMODULE_PARAMS if ke   │
│ ❱ 794 │   │   self.datamodule = self._setupDatamodule(FO_Dataset, self.datamoduleParams)         │
│   795 │   │                                                                                      │
│   796 │   │   self.adjustOutputPath()                                                            │
│   797                                                                                            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'tiling' is not defined